# BOOK REDDIT SENTIMENT ENRICHMENT 

## Description
Now we will enhance the book recommendations generated by both our within-domain and cross-domain recommender models, taking scores from our models and integrating sentiment analysis of posts/comments mentioning these books from Reddit. This results in a system that reorders the recommendations for each user based on Reddit-enriched ranking and, thus, takes the public opinion. We limit our search to four subreddits that are dedicated to books to optimize our search and decrease the noise found (for books we chose subreddits 'books', 'suggestmeabook', 'booksuggestions', 'bookclub'). 


# Set Up

In [1]:
# Import the findspark module 
import findspark

# Initialize via the full spark path
findspark.init("/usr/local/spark/")

In [2]:
from pyspark.sql import SparkSession
import collections
from pyspark.sql.functions import col
from pyspark.sql.functions import from_json, col, explode, struct, collect_list
from pyspark.sql.types import ArrayType, DoubleType
from pyspark.sql import Row

# Gets an existing :class:`SparkSession` or, if there is no existing one, creates a
# new one based on the options set in this builder.
spark = SparkSession.builder \
   .master("local[8]") \
   .appName("Book-Reddit-Enrichment") \
   .getOrCreate()

# Main entry point for Spark functionality. A SparkContext represents the
# connection to a Spark cluster, and can be used to create :class:`RDD` and
# broadcast variables on that cluster.
sc = spark.sparkContext

---
## Reddit Set up 
Load the necessary packages and connect to Reddit API with our credentials 

!! Credentials (accessAPI and credentials) have to be created as separate python objects and imported !!

In [3]:
import sys
!{sys.executable} -m pip install praw --no-cache-dir
!{sys.executable} -m pip install -U textblob --no-cache-dir

Requirement already up-to-date: textblob in /opt/conda/lib/python3.8/site-packages (0.18.0.post0)


In [4]:
# Import reddit api wrapper and other neccessary packages:
import praw
from textblob import TextBlob
import json
from datetime import datetime
from kafka import KafkaProducer
import re
import time

# Import passwords and tokens:
import credentials
import accessAPI

# connect to api with credentials received from the website and user acc 
def reddit_connection():
    try: 
        reddit = praw.Reddit(client_id= accessAPI.CLIENT_ID,
                     client_secret=accessAPI.SECRET, 
                     password=credentials.password, 
                     username=credentials.username,
                     user_agent="BookRecommendationSystem")
        return reddit

    except Exception as e:
        print("Error connecting to Reddit API")
        return None

---
## Reddit Data Collection
### Define helper functions that will be used later

In [5]:
def is_book_mentioned(text, book_title):
    # Checks if a specific book is mentioned in text
    if not text or not book_title:
        return False
    
    text_lower = text.lower()
    title_lower = book_title.lower()
    
    # Direct match
    if title_lower in text_lower:
        return True
    
    # Match without punctuation
    clean_title = re.sub(r'[^\w\s]', '', title_lower)
    clean_text = re.sub(r'[^\w\s]', '', text_lower)
    
    return clean_title in clean_text

def find_mentioned_books(text, book_titles):
    # Find which books from a list are mentioned in text
    mentioned = []
    for book in book_titles:
        if is_book_mentioned(text, book):
            mentioned.append(book)
    return mentioned

def postData(post, book_title, source_type):
    # Extract data from a Reddit post
    return {
        'type': 'post',
        'source': source_type,
        'title': post.title,
        'text': f"{post.title} {getattr(post, 'selftext', '')}",
        'score': post.score,
        'num_comments': post.num_comments,
        'created_utc': post.created_utc,
        'url': f"https://reddit.com{post.permalink}",
        'mentioned_books': [book_title],
        'subreddit': post.subreddit.display_name
    }

def commentData(comment, original_post, mentioned_books):
    # Extract data from a Reddit comment
    return {
        'type': 'comment',
        'source': 'comment',
        'title': f"Comment on: {original_post['title']}",
        'text': comment.body,
        'score': comment.score,
        'num_comments': 0,
        'created_utc': getattr(comment, 'created_utc', 0),
        'url': f"https://reddit.com{comment.permalink}",
        'mentioned_books': mentioned_books,
        'subreddit': original_post['subreddit']
    }

def process_posts_books(posts, book_titles, mentions, source_type):
    # Process a list of posts to find the book
    for post in posts:
        try:
            full_text = f"{post.title} {getattr(post, 'selftext', '')}"

            mentioned_books = find_mentioned_books(full_text, book_titles)
            
            if mentioned_books:
                for book in mentioned_books:
                    post_data = postData(post, book, source_type)
                    
                    if book not in mentions:
                        mentions[book] = []

                    mentions[book].append(post_data)
                    
        except Exception as e:
            continue

### Function to find Reddit posts mentioning book titles
Look for posts mentioning the given book title from the defined subreddits (in our case they will be 'suggestmeabook', 'booksuggestions', 'bookclub'). We search for the original title of the book taken from the dataset, as well as different possible variations of the name, as sometimes the original title is very unlikely to be used on Reddit. E.g. it might not find much for title 'City of Bones (The Mortal Instruments, #1)', so we look for other alternatives too, like 'City of Bones', 'The Mortal Instruments, #1', 'The Mortal Instruments 1', 'City of Bones The Mortal Instruments, #1' and others. This increases our chances of finding the necessary data. 

In [6]:
# Find Reddit posts with book_title mentioned and save data about them (text, score-popularity of the post, etc.): 

def book_post(reddit, book_title, subreddits):
    # Directly search for each book title in the reddit posts

    all_posts = []
    seen_post_ids = set()  # Track unique post IDs to avoid duplicates
    MAX_TOTAL_POSTS = 150  # Stop once this limit is hit

    # Clean the titles of the books and create variations of the title to look for:
    search_titles = [
        book_title,
        re.sub(r'\s*\([^)]*\)', '', book_title),  # Remove parentheses
        book_title.replace(':', ''), # Remove colons
        book_title.split(':')[0] if ':' in book_title else book_title, # First part if series
        re.sub(r'[^\w\s]', '', book_title) # Remove punctuation  
    ]
    
    # Add second part of the title after colon to search titles (if there's : in the title) 
    # E.g. 'Evicted: Poverty and Profit in the American City' -> look for 'Poverty and Profit in the American City' too
    if ':' in book_title:
        parts = book_title.split(':', 1)
        if len(parts) > 1 and parts[1].strip():
            search_titles.append(parts[1].strip())

    # If title starts with "The", add version without it (it's common in casual references: e.g. 'Hunger Games' instead of 'The Hunger Games'):
    if book_title.lower().startswith('the '):
        title_without_the = book_title[4:]  # Remove "The "
        search_titles.append(title_without_the)

    # Add content inside parentheses to the search as well (with different variations):
    # This could be helpful for finding data for titles like 'Harry Potter and the Deathly Hallows (Harry Potter, #7)', which are present in our dataset
    parenth_match = re.search(r'\(([^)]+)\)', book_title)
    if parenth_match:
        parenth_content = parenth_match.group(1)
        search_titles.append(parenth_content)  # Original parentheses content
        
        # For series-like content (e.g., "(Harry Potter, #7)")
        if '#' in parenth_content:
            # Remove the #: e.g. "Harry Potter, #7" -> "Harry Potter, 7"
            no_hash = parenth_content.replace('#', '')
            search_titles.append(no_hash)
            
            # Remove comma and #: e.g. "Harry Potter, #7" -> "Harry Potter 7"
            no_comma_hash = parenth_content.replace(',', '').replace('#', '')
            search_titles.append(no_comma_hash)
        
        # Remove punctuation from parentheses content
        parenth_no_punct = re.sub(r'[^\w\s]', '', parenth_content)
        if parenth_no_punct != parenth_content:
            search_titles.append(parenth_no_punct)
            
    
    # Remove any duplicates or empty terms
    search_titles = list(set([term.strip() for term in search_titles if term.strip()]))

    for name in subreddits:
        if len(all_posts) >= MAX_TOTAL_POSTS:
            print(f"Reached maximum limit of {MAX_TOTAL_POSTS} posts. Stopping search.")
            break
        try:
            subreddit = reddit.subreddit(name)
            
            for search_title in search_titles:
                if len(all_posts) >= MAX_TOTAL_POSTS:
                    print(f"Reached maximum limit of {MAX_TOTAL_POSTS} posts. Stopping search.")
                    break
                try:
                    posts_for_title = 0 # counter of the number of posts collected for a title (for convenience)
                    
                    # Looks for posts with this book title
                    results = subreddit.search(
                        search_title, 
                        limit=100,
                        sort='relevance'
                    )
                    for post in results:
                        if len(all_posts) >= MAX_TOTAL_POSTS:
                            print(f"Reached maximum limit of {MAX_TOTAL_POSTS} posts. Stopping search.")
                            break

                        # Check if we've already seen this post ID
                        if post.id in seen_post_ids:
                            continue  # Skip duplicate post
                        
                        # Add post ID to seen set
                        seen_post_ids.add(post.id)

                        # using getattr ensures that if a post for some reason doesn't have selftext, no errors arise: 
                        text = f"{post.title} {getattr(post, 'selftext', '')}"  
                        post_data = postData(post, book_title, 'search')
                        all_posts.append(post_data)
                        posts_for_title += 1
                    print(f"Found {posts_for_title} posts for '{search_title}' in r/{name}")
                    
                    time.sleep(1)

                except Exception as e:
                    print(f"Search failed for '{search_title}': {e}")
                    continue

        except Exception as e:
            print(f"Cannot access r/{name}: {e}")
            continue

    print(f"Total posts found in the search: {len(all_posts)}")
    return all_posts

### Function to find Reddit comments mentioning book titles
Look for comments to the posts collected previously and collect the ones mentioning our book. Once again we use use the same variations of the book title to broaden our search and, hopefully, collect more data. 

In [7]:
# Collect comments from the relevant posts:
def book_comments(reddit, posts_data, book_title):
    
    # Generate the same title variations as in book_post():
    search_titles = [
        book_title,
        re.sub(r'\s*\([^)]*\)', '', book_title),  # Remove parentheses
        book_title.replace(':', ''), # Remove colons
        book_title.split(':')[0] if ':' in book_title else book_title, # First part if series
        re.sub(r'[^\w\s]', '', book_title) # Remove punctuation  
    ]
    # Add second part of the title after colon to search titles (if there's : in the title) 
    # E.g. 'Evicted: Poverty and Profit in the American City' -> look for 'Poverty and Profit in the American City' too
    if ':' in book_title:
        parts = book_title.split(':', 1)
        if len(parts) > 1 and parts[1].strip():
            search_titles.append(parts[1].strip())

    # If title starts with "The", add version without it (it's common in casual references: e.g. 'Hunger Games' instead of 'The Hunger Games'):
    if book_title.lower().startswith('the '):
        title_without_the = book_title[4:]  # Remove "The "
        search_titles.append(title_without_the)

    # Add content inside parentheses to the search as well (with different variations):
    # This could be helpful for finding data for titles like 'Harry Potter and the Deathly Hallows (Harry Potter, #7)', which are present in our dataset
    parenth_match = re.search(r'\(([^)]+)\)', book_title)
    if parenth_match:
        parenth_content = parenth_match.group(1)
        search_titles.append(parenth_content)  # Original parentheses content
        
        # For series-like content (e.g., "(Harry Potter, #7)")
        if '#' in parenth_content:
            # Remove the #: e.g. "Harry Potter, #7" -> "Harry Potter, 7"
            no_hash = parenth_content.replace('#', '')
            search_titles.append(no_hash)
            
            # Remove comma and #: e.g. "Harry Potter, #7" -> "Harry Potter 7"
            no_comma_hash = parenth_content.replace(',', '').replace('#', '')
            search_titles.append(no_comma_hash)
        
        # Remove punctuation from parentheses content
        parenth_no_punct = re.sub(r'[^\w\s]', '', parenth_content)
        if parenth_no_punct != parenth_content:
            search_titles.append(parenth_no_punct)

    # Remove any duplicates or empty terms
    search_titles = list(set([term.strip() for term in search_titles if term.strip()]))
    
    # Add limits of the number of posts for which comments are collected, to avoid long processing and overloading it:
    if len(posts_data) > 40: # if there are more than 40 posts, use only the top 15 to collect comments:
        max_posts = 15
    else:
        max_posts = len(posts_data) # otherwise process all of them

    # Sort by scores to take the most popular comments (they would most probably have more comments):
    sorted_posts = sorted(posts_data, key=lambda x: x['score'], reverse=True)
    posts_to_process = sorted_posts[:max_posts]
    
    all_comments = []
    seen_comment_ids = set()  # Track unique comment IDs to avoid duplicates
    
    for post_data in posts_to_process:
        if len(all_comments) >= 50: # stop if we already collected at least 50 comments from other posts
            break 
        try:
            # Get the url of the post and load the comments:
            post = reddit.submission(url=post_data['url'])
            post.comments.replace_more(limit=3) # limit to 3 levels of threads in the comments
 
            comment_count_per_post = 0
            for comment in post.comments.list():
                if len(all_comments) >= 50: # stop if 50 comments reached
                    break 
                try:
                    # Check if the comment has a body and is longer than 10, so that it's meaningful:
                    if hasattr(comment, 'body') and len(comment.body) > 10:
                        # Check if we've already seen this comment ID
                        if comment.id in seen_comment_ids:
                            continue  # Skip duplicate comment
                        
                        # Check if it mentions any of the book title variations:
                        mentioned_books = find_mentioned_books(comment.body, search_titles)
                        if mentioned_books:
                            # Add comment ID to seen set
                            seen_comment_ids.add(comment.id)
                            
                            comment_data = commentData(comment, post_data, [book_title])  # Store original title
                            all_comments.append(comment_data)
                            comment_count_per_post += 1
                        if comment_count_per_post >= 50:
                            break
                except Exception as e:
                    continue

            time.sleep(1)

        except Exception as e:
            print(f"Failed to process comments: {e}")
            continue

    print(f"The number of comments collected for '{book_title}' is {len(all_comments)}")
    return all_comments

### Comprehensive Book Reddit Collection Function
Look for both posts and comments mentioning a given book title in the chosen subreddits. We are limiting our search to 150 posts and 50 comments maximum, to try and get enough meaningful data and to not exceed Reddit API limitations, as well as to decrease the time needed for making the final recommendations. 

In [8]:
# Function to collect Reddit posts and comments for multiple book titles
# Returns dictionary with book titles as keys and collected data as values
import builtins
def collect_reddit_book_data(reddit, book_titles, subreddits=None):
    # Use these subreddits, if others aren't given:
    if subreddits is None:
        subreddits = ['books', 'suggestmeabook', 'booksuggestions', 'bookclub']
    
    # Initialize results dictionary:
    results = {}
    
    for i, book_title in enumerate(book_titles, 1):
        print(f"\n--- Processing book {i}/{len(book_titles)}: '{book_title}' ---")
        
        # Step 1: Find posts
        all_posts = book_post(reddit, book_title, subreddits)
        
        # Step 2: Get comments from found posts
        all_comments = [] 
        if all_posts:
            all_comments = book_comments(reddit, all_posts, book_title)
        
        # Print collection results
        n_items = len(all_posts) + len(all_comments)
        print(f"Collection complete: {len(all_posts)} posts, {len(all_comments)} comments (total: {n_items})")
        
        # Store results for this book:
        results[book_title] = {
            'posts': all_posts,
            'comments': all_comments,
            'total_posts': len(all_posts),
            'total_comments': len(all_comments),
            'total_items': len(all_posts) + len(all_comments)
        }
    
    # Print summary
    print(f"\n=== COLLECTION SUMMARY ===")
    total_posts_all = builtins.sum([data['total_posts'] for data in results.values()])
    total_comments_all = builtins.sum([data['total_comments'] for data in results.values()])
    print(f"Total across all books: {total_posts_all} posts, {total_comments_all} comments")

    for book_title, data in results.items():
        print(f"'{book_title}': {data['total_posts']} posts, {data['total_comments']} comments")

    return results

---
# Sentiment Analysis
We will be using the TextBlob library for sentiment analysis and the parameter we'll focus on is polarity. It doesn't make sense to use subjectivity as what we're looking for are the subjective opinions of people. Polarity is exactly what we need in this case, as it captures the sentiment direction and strength, ranging from -1 (negative) to +1 (positive).

In [9]:
# Load additional necessary packages
import os
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType

### Define helper functions

In [10]:
# Define preprocessing helper function for the text:
def preprocess_text_spark(df, text_column):
    # Remove empty or null values:
    df = df.filter(col(text_column).isNotNull())
    df = df.filter(col(text_column) != "")
    
    # Remove URLs, mentions, hashtags, special characters and os on
    df = df.withColumn(text_column, F.regexp_replace(text_column, r'http\S+', ''))
    df = df.withColumn(text_column, F.regexp_replace(text_column, r'@\w+', ''))
    df = df.withColumn(text_column, F.regexp_replace(text_column, '#', '')) 
    df = df.withColumn(text_column, F.regexp_replace(text_column, ':', ''))
    df = df.withColumn(text_column, F.regexp_replace(text_column, r'\n', ' '))
    df = df.withColumn(text_column, F.regexp_replace(text_column, r'\*', ''))
    df = df.withColumn(text_column, F.regexp_replace(text_column, r'\s+', ' '))
    df = df.withColumn(text_column, F.trim(text_column))
    df = df.withColumn(text_column, F.lower(text_column))
    
    return df

# Define a polarity detection function using TextBlob:
def apply_sentiment_analysis(df, text_column):
    # Define a helper polarity detection function that returns the polarity score (from -1 to 1) using TextBlob:
    def polarity_detection(text):
        if text is None or text.strip() == "":
            return 0.0
        try:
            return float(TextBlob(text).sentiment.polarity)
        except:
            return 0.0
    
    # Create UDF for polarity detection (so that it can be applied to a PySpark dataframe with the data from Reddit)
    polarity_udf = udf(polarity_detection, FloatType())

    # Apply sentiment analysis to a PySpark dataframe with a text column:
    df = df.withColumn("polarity", polarity_udf(col(text_column)))
    return df

### Function to convert reddit data to Spark
We create a function that converts the Reddit data previously collected in the form of a nested dictionary structure containing posts and comments organized by book titles into PySpark DataFrames. It extracts and cleans text fields, filters out empty content, and standardizes the fields into rows with book_title, text, type, and score. This function helps prepare the data for sentiment analysis and using Spark makes it scalable, potentially enabling large amounts of data.

In [11]:
# Convert reddit data to Spark dataframes:
def process_reddit_data_to_spark(spark, sentiment_data):
    book_dataframes = {}
    
    for book_title, data in sentiment_data.items():
        # Combine posts and comments into a single list
        all_texts = []
        
        # Use field names from the structure defined in postData function:
        for post in data['posts']:
            if 'text' in post and post['text'] and post['text'].strip():
                all_texts.append(Row(
                    book_title=book_title,
                    text=post['text'].strip(),
                    type='post',
                    score=post.get('score', 0)
                ))
        
        # Use field names from the structure defined in commentData function:
        for comment in data['comments']:
            if 'text' in comment and comment['text'] and comment['text'] != '[removed]':
                all_texts.append(Row(
                    book_title=book_title,
                    text=comment['text'].strip(),
                    type='comment',
                    score=comment.get('score', 0)
                ))
        
        # Create Spark DataFrame:
        if all_texts:
            try:
                book_df = spark.createDataFrame(all_texts)
                book_dataframes[book_title] = book_df
            except Exception as e:
                print(f"Error creating DataFrame for {book_title}: {e}")
                book_dataframes[book_title] = None
        else:
            print(f"No valid text found for {book_title}")
            book_dataframes[book_title] = None
            
    return book_dataframes

In [12]:
# Function that will be used to take a single book's DataFrame and run the sentiment analysis on it:
def analyze_book_sentiment(spark, book_df):
    # If the dataframe doesn't exist, returns None:
    if book_df is None:
        return None
    
    # Preprocess text: clean it using the preprocess_text_spark defined earlier (removing URLs, mentions, hashtags, special characters, and so on)
    processed_df = preprocess_text_spark(book_df, 'text')
    
    # Apply sentiment analysis: use our UDF to add a 'polarity' column with sentiment scores (-1 to +1):
    sentiment_df = apply_sentiment_analysis(processed_df, 'text')
    
    # Calculate the metrics needed:
    metrics = sentiment_df.agg(
        F.avg('polarity').alias('avg_polarity'), # calculate the average polarity for a book
        F.count('polarity').alias('total_texts') # find the total number of post+comments analyzed
    ).collect()[0]
    
    # Returns overall sentiment and the number posts/comments analyzed for a book :
    return metrics 

## Combining model recommendations with sentiment analysis results

**Plan:**
- Step 1: Normalize polarity from [-1,1] to [0,1] 
- Step 2: Calculate volume multiplier based on post/comment counts 
- Step 3: Apply volume multiplier to normalized polarity 
- Step 4: Normalize Reddit scores to model score range, then combine using 0.6/0.4 weights
- Step 5: Penalize books with no Reddit data

**calculate_reddit_scores():** Get sentiment data for each book, normalizes polarity to [0,1] using: (polarity + 1) / 2, calculate volume multiplier based on text count (more texts = more reliable), return final Reddit score (normalized_polarity * volume_multiplier).

**calculate_volume_multiplier():** Assigns multipliers based on volume levels, 150+ texts = 1.0 (maximum reliability), down to 5 texts = 0.3 (minimum reliability). 

**combine_model_and_reddit_scores():** First normalize Reddit scores to the model score range, then combine using 0.6/0.4 (Models/Reddit) weights, apply the 0.05 penalty when no Reddit data exists, ensure scores don't go negative.


### Calculating the sentiment analysis coefficients

In [13]:
# Calculate volume multiplier based on number of posts/comments.
# Higher volume means higher multiplier because the analysis is more reliable.
def calculate_volume_multiplier(total_texts):
    if total_texts >= 150:  # Very high (close to max 200)
        return 1.0
    elif total_texts >= 100: 
        return 0.9
    elif total_texts >= 50:   
        return 0.8
    elif total_texts >= 20:   
        return 0.7
    elif total_texts >= 10:   
        return 0.6
    elif total_texts >= 5:    # Very low volume
        return 0.5
    else:  # Extremely low volume
        return 0.3

    
# Calculate Reddit sentiment scores for each book.
# Return scores from 0 to 1 where 1 is maximum.
def calculate_reddit_scores(sentiment_data, spark):
    # Convert to Spark DataFrames:
    book_dataframes = process_reddit_data_to_spark(spark, sentiment_data)
    
    reddit_scores = {}
    
    for book_title, book_df in book_dataframes.items():
        print()
        print(f"Processing {book_title}...")
        
        # Get sentiment metrics
        # If no data:
        if book_df is None:
            reddit_scores[book_title] = {
                'score': 0.0,  
                'total_texts': 0,
                'avg_polarity': 0.0,
                'volume_multiplier': 0.0
            }
            continue
            
        metrics = analyze_book_sentiment(spark, book_df)
        
        if metrics is None or metrics['total_texts'] == 0:
            reddit_scores[book_title] = {
                'score': 0.0,  
                'total_texts': 0,
                'avg_polarity': 0.0,
                'volume_multiplier': 0.0
            }
            continue
        
        # Extract values
        avg_polarity = float(metrics['avg_polarity'])
        total_texts = int(metrics['total_texts'])
        
        # Step 1: normalize polarity from [-1,1] to [0,1]:
        normalized_polarity = (avg_polarity + 1) / 2
        
        # Step 2: calculate volume multiplier based on popularity
        # Max possible: 200 texts (150 posts + 50 comments)
        volume_multiplier = calculate_volume_multiplier(total_texts)
        
        # Step 3: apply volume multiplier
        reddit_score = normalized_polarity * volume_multiplier
        
        reddit_scores[book_title] = {
            'score': reddit_score,
            'total_texts': total_texts,
            'avg_polarity': avg_polarity,
            'volume_multiplier': volume_multiplier
        }
        
        print(f"  Polarity: {avg_polarity:.3f} -> Normalized: {normalized_polarity:.3f}")
        print(f"  Volume: {total_texts} texts -> Multiplier: {volume_multiplier:.3f}")
        print(f"  Final Reddit score: {reddit_score:.3f}")
    
    return reddit_scores

### Combining the scores

In [14]:
# Combine model scores with Reddit sentiment scores
# no_data_penalty is the penalty that is subtracted when there's no data (to reflect that this book is not popular)
# We consider our models to be more reliable, so their weight it higher (0.6)
def combine_model_and_reddit_scores(model_recommendations, reddit_scores, 
                                  model_weight=0.6, reddit_weight=0.4, 
                                  no_data_penalty=0.05):
    combined_scores = []
    
    # Collect all Reddit scores to normalize them
    reddit_values = []
    model_values = []
    
    for book_title, model_score in model_recommendations:
        reddit_data = reddit_scores.get(book_title, {})
        reddit_score = reddit_data.get('score', 0.0)
        total_texts = reddit_data.get('total_texts', 0)
        
        if total_texts > 0:  # Only include books with Reddit data for normalization
            reddit_values.append(reddit_score)
            model_values.append(model_score) 
    
    
    # Calculate ranges
    if reddit_values:
        min_reddit = builtins.min(reddit_values)
        max_reddit = builtins.max(reddit_values)
        min_model = builtins.min(model_values)
        max_model = builtins.max(model_values)
        
        reddit_range = max_reddit - min_reddit
        model_range = max_model - min_model
        
        print(f"Reddit scores range: {min_reddit:.3f} to {max_reddit:.3f}")
        print(f"Model scores range: {min_model:.3f} to {max_model:.3f}")
    else:
        # No Reddit data available
        min_reddit = max_reddit = reddit_range = 0
        min_model = max_model = model_range = 0
        
        
    # Combine scores with normalized Reddit values:
    for book_title, model_score in model_recommendations:
        reddit_data = reddit_scores.get(book_title, {})
        reddit_score = reddit_data.get('score', 0.0)
        total_texts = reddit_data.get('total_texts', 0)
        
        if total_texts == 0:
            # No Reddit data: apply penalty to model score
            final_score = model_score - no_data_penalty
            print(f"{book_title}: No Reddit data, applying penalty of {no_data_penalty}")
        else:
            # Normalize Reddit score to model score range
            if reddit_range > 0:
                # Scale Reddit score to model range
                normalized_reddit = min_model + (reddit_score - min_reddit) / reddit_range * model_range
            else:
                # If no variation in Reddit scores, use middle of model range
                normalized_reddit = (min_model + max_model) / 2
                
            # Combine normalized scores
            final_score = (model_weight * model_score) + (reddit_weight * normalized_reddit)
            print(f"{book_title}: Model({model_score:.3f}) + Reddit({reddit_score:.3f} -> {normalized_reddit:.3f}) = {final_score:.3f}")
        
        # Ensure score doesn't go below 0
        final_score = builtins.max(0.0, final_score)

        combined_scores.append([book_title, final_score])
    
    # Sort by final score:
    combined_scores.sort(key=lambda x: x[1], reverse=True)
    
    return combined_scores

--- 
# Within-Domain
## Load recommendations from our models
The results were previously saved to a csv file (book_user_recs.csv).

In [15]:
# Load the CSV file
try: 
    recs_raw = spark.read.csv("../Models/models_results/book_user_recs.csv", header=True, inferSchema=True)
except Exception as e:
    print(f"Error loading file: {e}")
    raise
# Show a few rows to confirm
recs_raw.show(3)

+-------+---------------------------+
|user_id|hybrid_recommendations_json|
+-------+---------------------------+
|    148|       [[2963218.0,0.659...|
|    463|       [[1.3341979E7,0.8...|
|    471|       [[4.7815497E7,0.6...|
+-------+---------------------------+
only showing top 3 rows



In [16]:
# Loading book_metadata_old.json
try: 
    book_metadata = spark.read.json("../Models/within_domain_data/books_metadata_old.json")
except Exception as e:
    print(f"Error loading file: {e}")
    raise

In [17]:
book_metadata = book_metadata.select(
    col("item_id").cast("long"),  # cast item_id to match recs type
    col("title")
)
book_metadata.show(3)

+--------+--------------------+
| item_id|               title|
+--------+--------------------+
|16827462|The Fault in Our ...|
| 2792775|The Hunger Games ...|
| 8812783|Mockingjay (The H...|
+--------+--------------------+
only showing top 3 rows



In [18]:
# Define schema for JSON-like array field
recommendation_schema = ArrayType(ArrayType(DoubleType()))

# Parse JSON string column into nested array
recs_parsed = recs_raw.withColumn(
    "hybrid_recommendations",
    from_json(col("hybrid_recommendations_json"), recommendation_schema)
).drop("hybrid_recommendations_json")

# Filter out rows where parsing failed
recs_parsed = recs_parsed.filter(col("hybrid_recommendations").isNotNull())

# Explode recommendations: one row per recommended item
recs_exploded = recs_parsed.withColumn(
    "rec", explode("hybrid_recommendations")
).select(
    "user_id",
    col("rec").getItem(0).cast("long").alias("item_id"),  # cast ID to long
    col("rec").getItem(1).alias("score")
)

# Join with metadata to get titles
recs_with_titles = recs_exploded.join(
    book_metadata,
    on="item_id",
    how="left"
)

# Replace item_id with struct(title, score) instead of array
recs_with_titles = recs_with_titles.withColumn(
    "rec_with_title",
    struct("title", "score")
)

# Group back by user_id
final_user_recs_with_titles = recs_with_titles.groupBy("user_id").agg(
    collect_list("rec_with_title").alias("hybrid_recommendations_with_titles")
)

---
# Applying sentiment analysis to users
We'll be  demonstrating the Reddit sentiment enrichment on recommendations for 3 users, but with enough time and without the API limitations, the algorithm can be scaled up, and applied to any amount of users and recommendation.

In [19]:
# Show for 3 users with 15 recommended books each. The same can be done for the other users as well.
users = final_user_recs_with_titles.limit(3)  
users_data = users.collect()

for row in users_data:
    print(f"\nUser ID: {row['user_id']}")
    print("Top 15 Recommendation:")
    for title_score in row['hybrid_recommendations_with_titles'][:15]:
        print(f"  - {title_score['title']}: {title_score['score']:.4f}")

print()
print(f"=== REDDIT SENTIMENT ANALYSIS ===")
print(f"Processing {len(users_data)} user for demonstration...")

# Initialize Reddit connection (using your existing function)
reddit = reddit_connection()
if reddit is None:
    print("Failed to connect to Reddit API")
    exit()


User ID: 148
Top 15 Recommendation:
  - Harry Potter and the Deathly Hallows (Harry Potter, #7): 0.6591
  - Born a Crime: Stories From a South African Childhood: 0.6490
  - Dear Ijeawele, or a Feminist Manifesto in Fifteen Suggestions: 0.6434
  - Harry Potter and the Goblet of Fire (Harry Potter, #4): 0.6404
  - March: Book Three (March, #3): 0.6390
  - The Hate U Give: 0.6380
  - Quintana of Charyn (Lumatere Chronicles, #3): 0.6375
  - Just Mercy: A Story of Justice and Redemption: 0.6345
  - Fool's Fate (Tawny Man, #3): 0.6344
  - Hamilton: The Revolution: 0.6331
  - Locke & Key, Vol. 6: Alpha & Omega: 0.6330
  - Humans of New York: Stories: 0.6327
  - Mother Bruce: 0.6317
  - Froi of the Exiles (Lumatere Chronicles, #2): 0.6310
  - Harry Potter and the Prisoner of Azkaban (Harry Potter, #3): 0.6303

User ID: 463
Top 15 Recommendation:
  - Daytripper: 0.8556
  - Fool's Fate (Tawny Man, #3): 0.7895
  - Totto-chan: The Little Girl at the Window: 0.7748
  - The Fire Next Time: 0.7668
 

### Collect Reddit Data

In [20]:
all_book_titles = []
for user_row in users_data:
    for rec in user_row['hybrid_recommendations_with_titles']:
        if rec['title'] is not None and rec['title'] not in all_book_titles:
            all_book_titles.append(rec['title'])

# Collect Reddit data once for all books
reddit_data = collect_reddit_book_data(
    reddit,
    list(all_book_titles)
)


--- Processing book 1/38: 'Harry Potter and the Deathly Hallows (Harry Potter, #7)' ---
Found 100 posts for 'Harry Potter, #7' in r/books
Found 0 posts for 'Harry Potter 7' in r/books
Found 9 posts for 'Harry Potter and the Deathly Hallows (Harry Potter, #7)' in r/books
Found 0 posts for 'Harry Potter, 7' in r/books
Reached maximum limit of 150 posts. Stopping search.
Found 41 posts for 'Harry Potter and the Deathly Hallows' in r/books
Reached maximum limit of 150 posts. Stopping search.
Reached maximum limit of 150 posts. Stopping search.
Total posts found in the search: 150
The number of comments collected for 'Harry Potter and the Deathly Hallows (Harry Potter, #7)' is 10
Collection complete: 150 posts, 10 comments (total: 160)

--- Processing book 2/38: 'Born a Crime: Stories From a South African Childhood' ---
Found 66 posts for 'Stories From a South African Childhood' in r/books
Reached maximum limit of 150 posts. Stopping search.
Found 84 posts for 'Born a Crime' in r/books
Rea

### Example of data collected for the first book for each user

In [21]:
# Preview some sample Reddit data for the first book from each user
for user_row in users_data:
    user_id = user_row['user_id']
    
    # Get first book with a title for this user
    first_book_title = None
    for rec in user_row['hybrid_recommendations_with_titles']:
        if rec['title'] is not None:
            first_book_title = rec['title']
            break
    
    if first_book_title and first_book_title in reddit_data:
        print(f"\n=== Raw Data for USER {user_id}: '{first_book_title}' ===")
        book_data = reddit_data[first_book_title]
        print(f"\n-- Raw Posts --")
        for i, post in enumerate(book_data['posts'][:1], 1):
            print(f"\nPost #{i}")
            print(f"Type       : {post['type']}")
            print(f"Source     : {post['source']}")
            print(f"Title      : {post['title']}")
            print(f"Score      : {post['score']}")
            print(f"Comments   : {post['num_comments']}")
            print(f"Created    : {post['created_utc']}")
            print(f"Subreddit  : {post['subreddit']}")
            print(f"URL        : {post['url']}")
            print(f"Mentioned  : {post['mentioned_books']}")
            print(f"Text:\n{post['text'][:1500] + '...' if len(post['text']) > 1500 else post['text']}")
            print("-" * 80)
        print(f"\n-- Raw Comments --")
        for i, comment in enumerate(book_data['comments'][:1], 1):
            print(f"\nComment #{i}")
            print(f"Type       : {comment['type']}")
            print(f"Source     : {comment['source']}")
            print(f"Title      : {comment['title']}")
            print(f"Score      : {comment['score']}")
            print(f"Comments   : {comment['num_comments']}")
            print(f"Created    : {comment['created_utc']}")
            print(f"Subreddit  : {comment['subreddit']}")
            print(f"URL        : {comment['url']}")
            print(f"Mentioned  : {comment['mentioned_books']}")
            print(f"Text:\n{comment['text']}")
            print("-" * 80)


=== Raw Data for USER 148: 'Harry Potter and the Deathly Hallows (Harry Potter, #7)' ===

-- Raw Posts --

Post #1
Type       : post
Source     : search
Title      : Discworld Series > Harry Potter. Fight me.
Score      : 1933
Comments   : 752
Created    : 1600145597.0
Subreddit  : books
URL        : https://reddit.com/r/books/comments/it22dr/discworld_series_harry_potter_fight_me/
Mentioned  : ['Harry Potter and the Deathly Hallows (Harry Potter, #7)']
Text:
Discworld Series > Harry Potter. Fight me. The world building is fantastic in both—perhaps JK’s best skill as an author. (Edit: changed my mind on this, see other comments below.)

And while some of her characters are iconic, many (the non-white ones in particular) are inexcusably flat. 

Terry built a world that included all sorts and never wrote a single character that wasn’t fully realized. I mean THE SUITCASE had more personality than Cho or the Patels. 

His plots were wonderfully layered and used historic references and set

### Compute Reddit scores and combine with Model scores

In [22]:
# Compute Reddit scores (for sentiment analysis) using the data collected in reddit_data:
reddit_scores_cache = calculate_reddit_scores(
    reddit_data,
    spark
)


Processing Harry Potter and the Deathly Hallows (Harry Potter, #7)...
  Polarity: 0.141 -> Normalized: 0.570
  Volume: 160 texts -> Multiplier: 1.000
  Final Reddit score: 0.570

Processing Born a Crime: Stories From a South African Childhood...
  Polarity: 0.136 -> Normalized: 0.568
  Volume: 165 texts -> Multiplier: 1.000
  Final Reddit score: 0.568

Processing Dear Ijeawele, or a Feminist Manifesto in Fifteen Suggestions...
  Polarity: 0.227 -> Normalized: 0.613
  Volume: 22 texts -> Multiplier: 0.700
  Final Reddit score: 0.429

Processing Harry Potter and the Goblet of Fire (Harry Potter, #4)...
  Polarity: 0.146 -> Normalized: 0.573
  Volume: 151 texts -> Multiplier: 1.000
  Final Reddit score: 0.573

Processing March: Book Three (March, #3)...
  Polarity: 0.190 -> Normalized: 0.595
  Volume: 200 texts -> Multiplier: 1.000
  Final Reddit score: 0.595

Processing The Hate U Give...
  Polarity: 0.024 -> Normalized: 0.512
  Volume: 156 texts -> Multiplier: 1.000
  Final Reddit scor

In [23]:
# Process the user using the computed reddit_scores_cache and our combine_model_and_reddit_scores function:
def process_user_with_sentiment(user_id, user_recs, reddit_scores_cache):
    print(f"\n--- Processing User {user_id} ---")
    
    # Extract book titles from recommendations
    book_titles = []
    model_recommendations = []
    
    for rec in user_recs:
        if rec['title'] is not None:
            book_titles.append(rec['title'])
            model_recommendations.append((rec['title'], rec['score']))
    
    print(f"Found {len(book_titles)} books to analyze")
    print("Original 15 recommendations:")
    for i, (title, score) in enumerate(model_recommendations[:15]):
        print(f"  {i+1}. {title[:60]}... - Score: {score:.4f}")
    
    print("\nCombining model scores with Reddit sentiment analysis...")
    
    try:
        combined_scores = combine_model_and_reddit_scores(
            model_recommendations,
            reddit_scores_cache,
            model_weight=0.6,
            reddit_weight=0.4,
            no_data_penalty=0.05
        )
        return combined_scores
    except Exception as e:
        print(f"Error combining recommendations: {e}")
        return model_recommendations

# Final Recommendations - Within Domain

In [24]:
# Process users and print results using process_user_with_sentiment:
print("\n" + "="*80)
print("PROCESSING EXAMPLE USER")
print("="*80)
processed_users = []
for i, user_row in enumerate(users_data):
    user_id = user_row['user_id']
    user_recs = user_row['hybrid_recommendations_with_titles']
    
    try:
        # Use our function that reddit_scores_cache had already been collected
        sentiment_enhanced_recs = process_user_with_sentiment(
            user_id, 
            user_recs, 
            reddit_scores_cache 
        )
        
        # Store results:
        processed_users.append({
            'user_id': user_id,
            'original_recommendations': [(rec['title'], rec['score']) for rec in user_recs if rec['title'] is not None],
            'sentiment_enhanced_recommendations': sentiment_enhanced_recs
        })
        
        # Show comparison of previous scores with the final ones:
        print(f"\n{'='*60}")
        print(f"RESULTS FOR USER {user_id}")
        print(f"{'='*60}")
        
        print("\nTOP 15 COMPARISON:")
        print(f"{'Rank':<4} | {'Original Score':<13} | {'Enhanced Score':<13} | {'Change':<8} | {'Title'}")
        print("-" * 100)
        
        original_recs = processed_users[-1]['original_recommendations'][:15]
        enhanced_recs = sentiment_enhanced_recs[:15]
        
        for j in range(builtins.min(15, len(original_recs), len(enhanced_recs))):
            orig_score = original_recs[j][1]
            enh_score = enhanced_recs[j][1]
            change = enh_score - orig_score
            change_str = f"{change:+.4f}"
            title = enhanced_recs[j][0][:50] + "..." if len(enhanced_recs[j][0]) > 50 else enhanced_recs[j][0]
            
            print(f"{j+1:<4} | {orig_score:<13.4f} | {enh_score:<13.4f} | {change_str:<8} | {title}")
        
    except Exception as e:
        print(f"Error processing user {user_id}: {e}")
        continue
        

# Create PySpark DataFrame from results:

# Prepare data for PySpark DataFrame
spark_data = []
for user_data in processed_users:
    user_id = user_data['user_id']
    
    # Flatten original recommendations
    original_titles = [rec[0] for rec in user_data['original_recommendations']]
    original_scores = [rec[1] for rec in user_data['original_recommendations']]
    
    # Flatten enhanced recommendations
    enhanced_titles = [rec[0] for rec in user_data['sentiment_enhanced_recommendations']]
    enhanced_scores = [rec[1] for rec in user_data['sentiment_enhanced_recommendations']]
    
    spark_data.append({
        'user_id': user_id,
        'original_titles': original_titles,
        'original_scores': original_scores,
        'enhanced_titles': enhanced_titles,
        'enhanced_scores': enhanced_scores,
        'top_15_recommendations': enhanced_titles[:15] 
    })

# Define schema for better performance
schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("original_titles", ArrayType(StringType()), True),
    StructField("original_scores", ArrayType(FloatType()), True),
    StructField("enhanced_titles", ArrayType(StringType()), True),
    StructField("enhanced_scores", ArrayType(FloatType()), True),
    StructField("top_15_recommendations", ArrayType(StringType()), True)
])

# Create PySpark DataFrame
recommendations_df = spark.createDataFrame(spark_data, schema)

print("\nPySpark DataFrame Preview:")
recommendations_df.show(truncate=False)

print()
print("\n" + "="*80)
print("FINAL RECOMMENDATIONS ORDER")
print("="*80)

# Collect results
results = recommendations_df.collect()

for row in results:
    user_id = row['user_id']
    top_15 = row['top_15_recommendations']
    
    print(f"\nTOP 15 BOOK RECOMMENDATIONS FOR USER {user_id}")
    print("-" * 50)
    for i, title in enumerate(top_15, 1):
        print(f"{i:2d}. {title}")


PROCESSING EXAMPLE USER

--- Processing User 148 ---
Found 15 books to analyze
Original 15 recommendations:
  1. Harry Potter and the Deathly Hallows (Harry Potter, #7)... - Score: 0.6591
  2. Born a Crime: Stories From a South African Childhood... - Score: 0.6490
  3. Dear Ijeawele, or a Feminist Manifesto in Fifteen Suggestion... - Score: 0.6434
  4. Harry Potter and the Goblet of Fire (Harry Potter, #4)... - Score: 0.6404
  5. March: Book Three (March, #3)... - Score: 0.6390
  6. The Hate U Give... - Score: 0.6380
  7. Quintana of Charyn (Lumatere Chronicles, #3)... - Score: 0.6375
  8. Just Mercy: A Story of Justice and Redemption... - Score: 0.6345
  9. Fool's Fate (Tawny Man, #3)... - Score: 0.6344
  10. Hamilton: The Revolution... - Score: 0.6331
  11. Locke & Key, Vol. 6: Alpha & Omega... - Score: 0.6330
  12. Humans of New York: Stories... - Score: 0.6327
  13. Mother Bruce... - Score: 0.6317
  14. Froi of the Exiles (Lumatere Chronicles, #2)... - Score: 0.6310
  15. Harry Po

---
# Cross-Domain – Books Based on Movies

### Load the data 
Load the data recommendations from our cross-domain recommender models, which was previously saved as a json file.

In [25]:
try: 
    movies_to_books = spark.read.json("../Models/models_results/movies_to_books.json")
except Exception as e:
    print(f"Error loading file: {e}")
    raise
movies_to_books.show(n=3, truncate=False)

+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|movie_id|recommended_books                                                                                                                                                                                                                                                                                                                  |
+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [26]:
# Loading movie metadata:
try: 
    movie_metadata = spark.read.csv("../Models/within_domain_data/movies_metadata_filtered_1980_plus.csv", header = True)
except Exception as e:
    print(f"Error loading file: {e}")
    raise

movie_metadata.show(3)

+---------+-------------+-------+-------+--------------------+----------------+----+
|avgRating|   directedBy| imdbId|item_id|            starring|           title|year|
+---------+-------------+-------+-------+--------------------+----------------+----+
|  3.89146|John Lasseter|0114709|      1|Tim Allen, Tom Ha...|       Toy Story|1995|
|  3.26605| Joe Johnston|0113497|      2|Jonathan Hyde, Br...|         Jumanji|1995|
|  3.17146|Howard Deutch|0113228|      3|Jack Lemmon, Walt...|Grumpier Old Men|1995|
+---------+-------------+-------+-------+--------------------+----------------+----+
only showing top 3 rows



In [27]:
movies_exploded = movies_to_books.withColumn("rec", explode("recommended_books")) \
    .select("movie_id", 
            col("rec.book_id").alias("book_item_id"), 
            col("rec.score").alias("score"))

# Join both metadata tables and group back
movie_books = movies_exploded \
    .join(book_metadata.select("item_id", col("title").alias("book_title")), 
          movies_exploded.book_item_id == book_metadata.item_id, "left") \
    .join(movie_metadata.select("item_id", col("title").alias("movie_title")), 
          movies_exploded.movie_id == movie_metadata.item_id, "left") \
    .select("movie_title", "book_title", "score") \
    .groupBy("movie_title") \
    .agg(collect_list(struct("book_title", "score")).alias("recommended_books_with_titles"))

movie_books.show(3, truncate=False)

+----------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|movie_title                             |recommended_books_with_titles                                                                                                                                                                                                                                                             

---
## Applying sentiment analysis to books recommended based on movies

In [28]:
# Show for 3 movies with 10 recommended books each. The same can be done for the other movies as well.
movies = movie_books.limit(3)  
movies_data = movies.collect()

for row in movies_data:
    print(f"\nMovie Title: {row['movie_title']}")
    print("Top 10 Book Recommendations:")
    for title_score in row['recommended_books_with_titles'][:10]:
        print(f"  - {title_score['book_title']}: {title_score['score']:.4f}")
        
print()
print(f"=== REDDIT SENTIMENT ANALYSIS ===")
print(f"Processing {len(movies_data)} movies for demonstration...")


Movie Title: Sugar & Spice
Top 10 Book Recommendations:
  - City of Bones (The Mortal Instruments, #1): 0.3503
  - Demon from the Dark (Immortals After Dark #10): 0.3483
  - Lord of Shadows (The Dark Artifices, #2): 0.3477
  - The Umbrella Academy, Vol. 1: The Apocalypse Suite: 0.3477
  - The Black Prism (Lightbringer, #1): 0.3476
  - Red-Headed Stepchild (Sabina Kane, #1): 0.3475
  - A Discovery of Witches (All Souls Trilogy, #1): 0.3474
  - The Selection (The Selection, #1): 0.3473
  - Missing Pieces: 0.3473
  - Midnight Alley (The Morganville Vampires, #3): 0.3471

Movie Title: Skin I Live In, The (La piel que habito)
Top 10 Book Recommendations:
  - The Shade of the Moon (Last Survivors, #4): 0.2255
  - Ghostgirl (Ghostgirl, #1): 0.2254
  - The Supernatural Enhancements: 0.2248
  - The Interpretation of Murder (Freud, #1): 0.2247
  - The Secret Scripture: 0.2240
  - The Engagements: 0.2233
  - 2312: 0.2231
  - The Taking: 0.2225
  - The Master Magician (The Paper Magician Trilogy,

### Collect Reddit Data

In [29]:
all_book_titles_movie = []
for movie_row in movies_data:
    for rec in movie_row['recommended_books_with_titles']:
        if rec['book_title'] is not None and rec['book_title'] not in all_book_titles_movie:
            all_book_titles_movie.append(rec['book_title'])

# Collect Reddit data once for all books
reddit_data_movie = collect_reddit_book_data(
    reddit,
    list(all_book_titles_movie)
)


--- Processing book 1/30: 'City of Bones (The Mortal Instruments, #1)' ---
Found 5 posts for 'The Mortal Instruments, 1' in r/books
Found 84 posts for 'City of Bones' in r/books
Found 22 posts for 'City of Bones The Mortal Instruments 1' in r/books
Found 0 posts for 'City of Bones (The Mortal Instruments, #1)' in r/books
Found 0 posts for 'The Mortal Instruments 1' in r/books
Found 0 posts for 'The Mortal Instruments, #1' in r/books
Found 26 posts for 'The Mortal Instruments, 1' in r/suggestmeabook
Reached maximum limit of 150 posts. Stopping search.
Found 13 posts for 'City of Bones' in r/suggestmeabook
Reached maximum limit of 150 posts. Stopping search.
Reached maximum limit of 150 posts. Stopping search.
Total posts found in the search: 150
The number of comments collected for 'City of Bones (The Mortal Instruments, #1)' is 6
Collection complete: 150 posts, 6 comments (total: 156)

--- Processing book 2/30: 'Demon from the Dark (Immortals After Dark #10)' ---
Found 3 posts for 'De

In [30]:
# Compute Reddit scores (for sentiment analysis) using the data collected in reddit_data_movie:
reddit_scores_cache_movie = calculate_reddit_scores(
    reddit_data_movie,
    spark
)


Processing City of Bones (The Mortal Instruments, #1)...
  Polarity: 0.132 -> Normalized: 0.566
  Volume: 156 texts -> Multiplier: 1.000
  Final Reddit score: 0.566

Processing Demon from the Dark (Immortals After Dark #10)...
  Polarity: 0.098 -> Normalized: 0.549
  Volume: 150 texts -> Multiplier: 1.000
  Final Reddit score: 0.549

Processing Lord of Shadows (The Dark Artifices, #2)...
  Polarity: 0.130 -> Normalized: 0.565
  Volume: 150 texts -> Multiplier: 1.000
  Final Reddit score: 0.565

Processing The Umbrella Academy, Vol. 1: The Apocalypse Suite...
  Polarity: 0.118 -> Normalized: 0.559
  Volume: 32 texts -> Multiplier: 0.700
  Final Reddit score: 0.391

Processing The Black Prism (Lightbringer, #1)...
  Polarity: 0.161 -> Normalized: 0.581
  Volume: 158 texts -> Multiplier: 1.000
  Final Reddit score: 0.581

Processing Red-Headed Stepchild (Sabina Kane, #1)...
  Polarity: 0.118 -> Normalized: 0.559
  Volume: 52 texts -> Multiplier: 0.800
  Final Reddit score: 0.447

Process

In [31]:
# Process the user using the computed reddit_scores_cache and our combine_model_and_reddit_scores function:
def process_movie_with_sentiment(movie_title, recommended_books, reddit_scores_cache_movie):
    print(f"\n--- Processing Movie: {movie_title} ---")
    
    # Extract book titles from recommendations
    book_titles = []
    model_recommendations = []
    
    for rec in recommended_books:
        if rec['book_title'] is not None:
            book_titles.append(rec['book_title'])
            model_recommendations.append((rec['book_title'], rec['score']))
    
    print(f"Found {len(book_titles)} books to analyze")
    print("Original 10 recommendations:")
    for i, (title, score) in enumerate(model_recommendations[:15]):
        print(f"  {i+1}. {title[:60]}... - Score: {score:.4f}")
    
    print("\nCombining model scores with Reddit sentiment analysis...")
    
    try:
        combined_scores = combine_model_and_reddit_scores(
            model_recommendations,
            reddit_scores_cache_movie,
            model_weight=0.6,
            reddit_weight=0.4,
            no_data_penalty=0.05
        )
        return combined_scores
    except Exception as e:
        print(f"Error combining recommendations: {e}")
        return model_recommendations

# Final Recommendations - Cross Domain

In [32]:
# Process for each movie and print results using process_movie_with_sentiment:
print("\n" + "="*80)
print("PROCESSING EXAMPLE MOVIES")
print("="*80)
processed_movies = []

for i, movie_row in enumerate(movies_data):
    movie_title = movie_row['movie_title']
    recommended_books = movie_row['recommended_books_with_titles']
    
    try:
        sentiment_enhanced_recs = process_movie_with_sentiment(
            movie_title,
            recommended_books,
            reddit_scores_cache_movie
        )
        
        # Store results:
        processed_movies.append({
            'movie_title': movie_title,
            'original_recommendations': [(rec['book_title'], rec['score']) for rec in recommended_books if rec['book_title'] is not None],
            'sentiment_enhanced_recommendations': sentiment_enhanced_recs
        })
        
        # Show comparison of previous scores with the final ones:
        print(f"\n{'='*60}")
        print(f"RESULTS FOR MOVIE: {movie_title}")
        print(f"{'='*60}")
        
        print("\nTOP 10 COMPARISON:")
        print(f"{'Rank':<4} | {'Original Score':<13} | {'Enhanced Score':<13} | {'Change':<8} | {'Title'}")
        print("-" * 100)
        
        original_recs = processed_movies[-1]['original_recommendations'][:10]
        enhanced_recs = sentiment_enhanced_recs[:10]
        
        for j in range(builtins.min(10, len(original_recs), len(enhanced_recs))):
            orig_score = original_recs[j][1]
            enh_score = enhanced_recs[j][1]
            change = enh_score - orig_score
            change_str = f"{change:+.4f}"
            title = enhanced_recs[j][0][:50] + "..." if len(enhanced_recs[j][0]) > 50 else enhanced_recs[j][0]
            
            print(f"{j+1:<4} | {orig_score:<13.4f} | {enh_score:<13.4f} | {change_str:<8} | {title}")
        
    except Exception as e:
        print(f"Error processing movie {movie_title}: {e}")
        continue
        

# Create PySpark DataFrame from results:

# Prepare data for PySpark DataFrame
spark_data = []
for movie_data in processed_movies:
    movie_title = movie_data['movie_title']
    
    # Flatten original recommendations
    original_titles = [rec[0] for rec in movie_data['original_recommendations']]
    original_scores = [rec[1] for rec in movie_data['original_recommendations']]
    
    # Flatten enhanced recommendations
    enhanced_titles = [rec[0] for rec in movie_data['sentiment_enhanced_recommendations']]
    enhanced_scores = [rec[1] for rec in movie_data['sentiment_enhanced_recommendations']]
    
    spark_data.append({
        'movie_title': movie_title,
        'original_titles': original_titles,
        'original_scores': original_scores,
        'enhanced_titles': enhanced_titles,
        'enhanced_scores': enhanced_scores,
        'top_10_recommendations': enhanced_titles[:10]
    })

schema = StructType([
    StructField("movie_title", StringType(), True),
    StructField("original_titles", ArrayType(StringType()), True),
    StructField("original_scores", ArrayType(FloatType()), True),
    StructField("enhanced_titles", ArrayType(StringType()), True),
    StructField("enhanced_scores", ArrayType(FloatType()), True),
    StructField("top_10_recommendations", ArrayType(StringType()), True)
])

# Create PySpark DataFrame
cross_recommendations_df_books = spark.createDataFrame(spark_data, schema)
cross_recommendations_df_books.show(truncate=False)


print()
print("\n" + "="*80)
print("FINAL RECOMMENDATIONS ORDER")
print("="*80)

# Collect results 
results = cross_recommendations_df_books.collect()
for row in results:
    movie_title = row['movie_title']
    top_10 = row['top_10_recommendations']
    
    print(f"\nTOP 10 BOOK RECOMMENDATIONS FOR MOVIE: {movie_title}")
    print("-" * 50)
    for i, title in enumerate(top_10, 1):
        print(f"{i:2d}. {title}")


PROCESSING EXAMPLE MOVIES

--- Processing Movie: Sugar & Spice ---
Found 10 books to analyze
Original 10 recommendations:
  1. City of Bones (The Mortal Instruments, #1)... - Score: 0.3503
  2. Demon from the Dark (Immortals After Dark #10)... - Score: 0.3483
  3. Lord of Shadows (The Dark Artifices, #2)... - Score: 0.3477
  4. The Umbrella Academy, Vol. 1: The Apocalypse Suite... - Score: 0.3477
  5. The Black Prism (Lightbringer, #1)... - Score: 0.3476
  6. Red-Headed Stepchild (Sabina Kane, #1)... - Score: 0.3475
  7. A Discovery of Witches (All Souls Trilogy, #1)... - Score: 0.3474
  8. The Selection (The Selection, #1)... - Score: 0.3473
  9. Missing Pieces... - Score: 0.3473
  10. Midnight Alley (The Morganville Vampires, #3)... - Score: 0.3471

Combining model scores with Reddit sentiment analysis...
Reddit scores range: 0.389 to 0.581
Model scores range: 0.347 to 0.350
City of Bones (The Mortal Instruments, #1): Model(0.350) + Reddit(0.566 -> 0.350) = 0.350
Demon from the Dark